# Triangular L=6 HMC Small-Parameter Benchmark Notebook

Point `ROOT` to the directory containing `small_benchmark.json`, `small_benchmark_cases.csv`, `small_benchmark_observables.csv`, `small_benchmark_samples.csv`, and optionally `small_tune.csv`.


In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path("data/triangular_hmc_small_benchmark").resolve()
summary = json.loads((ROOT / "small_benchmark.json").read_text(encoding="utf-8"))
cases = pd.read_csv(ROOT / "small_benchmark_cases.csv")
obs = pd.read_csv(ROOT / "small_benchmark_observables.csv")
samples = pd.read_csv(ROOT / "small_benchmark_samples.csv")
tune = pd.read_csv(ROOT / "small_tune.csv") if (ROOT / "small_tune.csv").exists() else None
summary["passed"], cases.shape, obs.shape, samples.shape


In [ ]:
display(cases.sort_values(["Nbos", "U2"])[[
    "name", "Nbos", "U2", "nfrog", "hmc_dt", "hmc_jitter", "hmc_mass",
    "acceptance_mean", "local_tau_int_doubleOcc", "hmc_tau_int_doubleOcc",
    "local_ess_per_sec_doubleOcc", "hmc_ess_per_sec_doubleOcc", "passed"
]])


In [ ]:
pivot = cases.pivot(index="Nbos", columns="U2", values="passed").sort_index().sort_index(axis=1)
fig, ax = plt.subplots(figsize=(7, 3.5))
im = ax.imshow(pivot.to_numpy(dtype=float), aspect="auto", cmap="RdYlGn", vmin=0.0, vmax=1.0)
ax.set_xticks(np.arange(pivot.shape[1]), [f"{u:g}" for u in pivot.columns])
ax.set_yticks(np.arange(pivot.shape[0]), [f"{n:g}" for n in pivot.index])
ax.set_xlabel("U2")
ax.set_ylabel("Nbos")
ax.set_title("Strict benchmark pass matrix")
for (i, j), value in np.ndenumerate(pivot.to_numpy(dtype=float)):
    ax.text(j, i, "PASS" if value > 0.5 else "FAIL", ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax)
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
for nbos, group in cases.groupby("Nbos", sort=True):
    group = group.sort_values("U2")
    x = group["U2"].to_numpy()
    axes[0].errorbar(x, group["acceptance_mean"], yerr=group["acceptance_stderr"], marker="o", capsize=3, label=f"Nbos={nbos}")
    axes[1].plot(x, group["local_tau_int_doubleOcc"], marker="o", linestyle="-", label=f"Local N={nbos}")
    axes[1].plot(x, group["hmc_tau_int_doubleOcc"], marker="s", linestyle="--", label=f"HMC N={nbos}")
    axes[2].plot(x, group["speed_ratio_hmc_over_local"], marker="o", linestyle="-", label=f"Nbos={nbos}")
for ax in axes:
    ax.set_xscale("log")
    ax.set_xlabel("U2")
axes[0].set_ylabel("Acceptance")
axes[1].set_ylabel("tau_int(doubleOcc)")
axes[2].set_ylabel("ESS/sec ratio")
axes[0].legend(fontsize=8)
axes[1].legend(fontsize=7, ncol=2)
axes[2].legend(fontsize=8)
plt.show()


In [ ]:
curve_observables = ["IPR", "kinetic", "doubleOcc", "squareOcc", "nearestOcc", "SF_Gamma", "SF_K", "PF_Gamma", "C3_Gamma", "dentot_Gamma", "denden_Gamma"]
for obs_name in curve_observables:
    subset = obs[obs["observable"] == obs_name].copy()
    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    for nbos, group in subset.groupby("Nbos", sort=True):
        group = group.sort_values("U2")
        x = group["U2"].to_numpy()
        ax.errorbar(x, group["local_mean"], yerr=group["local_err"], marker="o", linestyle="-", capsize=3, label=f"Local N={nbos}")
        ax.errorbar(x, group["hmc_mean"], yerr=group["hmc_err"], marker="s", linestyle="--", capsize=3, label=f"HMC N={nbos}")
    ax.set_xscale("log")
    ax.set_xlabel("U2")
    ax.set_ylabel(obs_name)
    ax.set_title(f"{obs_name} vs U2")
    ax.legend(fontsize=8, ncol=2)
    plt.show()


In [ ]:
series_observables = ["IPR", "kinetic", "doubleOcc", "nearestOcc", "SF_Gamma", "SF_K"]
for (nbos, u2), case_df in samples.groupby(["Nbos", "U2"], sort=True):
    fig, axes = plt.subplots(len(series_observables), 1, figsize=(10, 2.7 * len(series_observables)), sharex=True, constrained_layout=True)
    for idx, obs_name in enumerate(series_observables):
        ax = axes[idx]
        subset = case_df[case_df["observable"] == obs_name].sort_values(["mode", "sample_index"])
        for mode, mode_df in subset.groupby("mode"):
            ax.plot(mode_df["sample_index"], mode_df["value"], label=mode.upper(), linewidth=1.1)
        ax.set_ylabel(obs_name)
        ax.legend(fontsize=8)
    axes[-1].set_xlabel("sample index after thermal cut")
    fig.suptitle(f"Nbos={nbos}, U2={u2:g}")
    plt.show()


In [ ]:
if tune is not None:
    for name, case_df in tune.groupby("name", sort=True):
        case_df = case_df.copy()
        case_df["traj_len"] = case_df["nfrog"] * case_df["hmc_dt"]
        fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
        for mass, mass_df in case_df.groupby("hmc_mass", sort=True):
            label = f"mass={mass:g}"
            mass_df = mass_df.sort_values("traj_len")
            axes[0].plot(mass_df["traj_len"], mass_df["acceptance_mean"], marker="o", label=label)
            axes[1].plot(mass_df["traj_len"], mass_df["tau_int_doubleOcc_mean"], marker="o", label=label)
            axes[2].plot(mass_df["traj_len"], mass_df["ess_per_sec_doubleOcc_mean"], marker="o", label=label)
        for ax in axes:
            ax.set_xscale("log")
            ax.set_xlabel("trajectory length")
            ax.legend(fontsize=8)
        axes[0].set_ylabel("Acceptance")
        axes[1].set_ylabel("tau_int(doubleOcc)")
        axes[2].set_ylabel("ESS/sec")
        fig.suptitle(name)
        plt.show()
